# Advanced RAG Pipeline




In [1]:
%pip install -q \
    langchain-qdrant langchain-openai langchain-community langchain-experimental \
    qdrant-client fastembed \
    sentence-transformers \
    ragas \
    pypdf python-dotenv tqdm pandas rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.

In [2]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from pathlib import Path

# Re-ranking
from sentence_transformers import CrossEncoder

# Ragas
from ragas.testset import TestsetGenerator
from ragas import evaluate, EvaluationDataset
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_5349/1451843628.py:13: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy
/tmp/ipykernel_5349/1451843628.py:13: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas

False

In [5]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')

In [6]:
pdf_path = Path("/content/MT RESEARCH FINAL.pdf")
loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"Loaded {len(docs)} pages")

Loaded 21 pages


### Semantic Chunking



In [7]:
text_splitter = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=1,
)
split_docs = text_splitter.split_documents(docs)
print(f"Created {len(split_docs)} chunks")
print(f"Avg chunk length: {sum(len(d.page_content) for d in split_docs) // len(split_docs)} chars")

Created 93 chunks
Avg chunk length: 466 chars


## Hybrid Vector Store
### vector + keyword search

In [9]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

vector_store = QdrantVectorStore.from_documents(
    documents=split_docs,
    url="https://9219ef2c-5862-409f-93ef-f0808298aad3.eu-west-2-0.aws.cloud.qdrant.io",
    api_key=QDRANT_API_KEY,
    collection_name="advanced_rag_final",
    sparse_embedding=sparse_embeddings,
    embedding=embedding_model,
    retrieval_mode=RetrievalMode.HYBRID,
)
print("Indexing complete — Qdrant hybrid (dense+sparse) vector store ready")

Indexing complete — Qdrant hybrid (dense+sparse) vector store ready


### Query Expansion (Multi-Query)



In [20]:
rewriter_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

def expand_query(query: str, n_variations: int = 3) -> list:
    prompt = f"""Generate {n_variations} different phrasings of the following query.
Rules:
- Keep all key terms and concepts intact
- Each variation should approach the topic from a different angle
- Vary the vocabulary while preserving the meaning
- Return ONLY the queries, one per line, no numbering or bullets.

Original query: {query}

Variations:"""
    response = rewriter_llm.invoke(prompt).content.strip()
    variations = [q.strip() for q in response.split("\n") if q.strip()]
    return variations[:n_variations]


test_query = "How is AI used in Indian tourism?"
print(f"Original:  {test_query}")
print("\nQuery variations:")
for i, v in enumerate(expand_query(test_query, n_variations=3), 1):
    print(f"  {i}. {v}")

Original:  How is AI used in Indian tourism?

Query variations:
  1. In what ways does AI contribute to the tourism industry in India?
  2. What are the applications of artificial intelligence within Indian tourism?
  3. How does artificial intelligence impact the tourism sector in India?


## Reciprocal Rank Fusion (RRF)

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}$$


In [38]:
from collections import defaultdict


def reciprocal_rank_fusion(results_lists: list, k: int = 60) -> list:
    """Merge multiple ranked lists. Returns [(doc, score), ...] sorted best first."""
    scores = defaultdict(float)
    doc_map = {}
    for results in results_lists:
        for rank, doc in enumerate(results):
            key = doc.page_content
            if key not in doc_map:
                doc_map[key] = doc
            scores[key] += 1 / (k + rank)
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [(doc_map[content], score) for content, score in fused]



### Cross-Encoder Reranking


In [22]:
reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank_documents(query: str, documents: list, top_k: int = 5) -> list:
    """Cross-encoder rerank. Returns [(doc, score), ...] sorted best first."""
    if not documents:
        return []
    pairs = [[query, doc.page_content] for doc in documents]
    scores = reranker_model.predict(pairs)
    return sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)[:top_k]

print("Reranker ready")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker ready


### Advanced RAG Pipeline

`query → expand → hybrid retrieve → RRF → cross-encoder rerank → LLM -> Output`.

In [13]:
def advanced_rag_query(
    query: str,
    k: int = 20,
    rerank_top_k: int = 5,
    use_rrf: bool = True,
    n_query_variations: int = 3,
    verbose: bool = True,
):
    if use_rrf:
        query_variations = [query] + expand_query(query, n_variations=n_query_variations)
        if verbose:
            print(f"\nQuery Expansion\n  Original: {query}")
            for i, q in enumerate(query_variations, 1):
                print(f"    {i}. {q}")
        all_results = [vector_store.similarity_search(q, k=k) for q in query_variations]
        fused_results = reciprocal_rank_fusion(all_results, k=60)
        retrieved_docs = [doc for doc, _ in fused_results]
        if verbose:
            print(f"\n[Hybrid Retrieval + RRF] fused {len(retrieved_docs)} unique docs from {len(query_variations)} variations")
    else:
        retrieved_docs = vector_store.similarity_search(query, k=k)
        if verbose:
            print(f"\n[Hybrid Retrieval] retrieved {len(retrieved_docs)} docs")

    if not retrieved_docs:
        return {"answer": "No relevant documents found.", "sources": [], "query_used": query}

    if verbose:
        print(f"\n[Re-ranking] top {rerank_top_k} from {len(retrieved_docs)} docs...")

    reranked_docs = rerank_documents(query, retrieved_docs, top_k=rerank_top_k)

    if verbose:
        print("  Re-ranked scores (top 3):")
        for i, (_, score) in enumerate(reranked_docs[:3], 1):
            print(f"    #{i}: score={score:.4f}")

    contexts = [doc.page_content for doc, _ in reranked_docs]
    context_text = "\n\n---\n\n".join(contexts)

    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    prompt = f"""You are an expert AI assistant for deep learning and machine learning.
Answer the question using ONLY the provided context below.

Instructions:
- Base your answer strictly on the context; do not add outside knowledge
- Be precise and thorough — cover all relevant aspects mentioned in the context
- Use clear structure (bullet points or paragraphs) when the answer has multiple parts
- If the context does not contain enough information to answer fully, say so explicitly

Context:
{context_text}

Question: {query}

Answer:"""
    response = llm.invoke(prompt).content

    if verbose:
        print(f"\n{'-'*50}\nANSWER:\n{response}\n{'-------'}")

    return {
        "answer": response,
        "sources": contexts,
        "query_used": query,
        "reranked_docs": reranked_docs,
    }


print("advanced_rag_query ready")

advanced_rag_query ready


In [23]:
#testing

_ = advanced_rag_query(
    query="What is backpropagation and how does it work?", # this question context is not present in my document
    # just checking and re-ranked scores were 0.0001 means working good.
    k=20,
    rerank_top_k=5,
    use_rrf=True,
    n_query_variations=3,
    verbose=True,
)


[Query Expansion]
  Original: What is backpropagation and how does it work?
    1. What is backpropagation and how does it work?
    2. Can you explain the concept of backpropagation and the mechanism behind its operation?
    3. How does backpropagation function, and what exactly does it entail?
    4. What does backpropagation mean, and in what way is it implemented?

[Hybrid Retrieval + RRF] fused 39 unique docs from 4 variations

[Re-ranking] top 5 from 39 docs...
  Re-ranked scores (top 3):
    #1: score=0.0001
    #2: score=0.0001
    #3: score=0.0001

ANSWER:
The provided context does not contain any information about backpropagation or how it works. Therefore, I cannot answer the question based on the given material.


### Generate Test Dataset (Ragas)

In [32]:
# generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
# generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

# generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# print("Generating test dataset... (calls OpenAI API)")
# dataset = generator.generate_with_langchain_docs(docs, testset_size=10)
# print(f"Generated {len(dataset)} test samples")
# dataset.to_pandas()

import pandas as pd

test_data = [
    {
        "user_input": "What was the projected CAGR of the global AI in travel market from 2024 to 2029?",
        "reference": "The global AI in travel market was projected to grow at a CAGR of approximately 33.8%, increasing from $123.72 billion in 2024 to an estimated $531.95 billion by 2029."
    },
    {
        "user_input": "How much has India pledged for AI investment?",
        "reference": "India pledged $1.25 billion for AI investment."
    },
    {
        "user_input": "How does Heathrow Airport's AI adoption compare to Delhi Airport, and what does this reveal about the broader gap between global and Indian aviation?",
        "reference": "Heathrow's AI chatbot Hallie resolves around 90% of queries without human intervention, while Delhi Airport still relies more on human staff despite having kiosks and mobile apps. This reflects the broader pattern where resource-rich international airports have fully automated customer service, while Indian airports are still scaling basic digital infrastructure."
    },
    {
        "user_input": "How have AI-driven personalization strategies evolved differently in India versus global platforms like Booking.com?",
        "reference": "Globally, Booking.com has long used mature AI algorithms with a large user base and released a Global AI Sentiment Report showing high traveler demand. In India, MakeMyTrip integrated OpenAI's GPT API more recently to enable chat-to-book experiences, rapidly adapting to multilingual Indian markets. While both use similar underlying AI, Booking.com benefits from earlier maturity whereas MakeMyTrip is catching up with localization as its edge."
    },
    {
        "user_input": "What AI robot did Japan showcase at Expo 2025 Osaka?",
        "reference": "Japan showcased Miraia Lynx, an AI concierge robot at Expo 2025 Osaka that provides multilingual assistance and health monitoring."
    },
    {
        "user_input": "What percentage of travelers were excited about AI according to Booking.com's survey?",
        "reference": "91% of travelers were excited about AI and 89% wanted to use AI for travel planning, however only 6% fully trusted AI and 91% had privacy concerns."
    },
    {
        "user_input": "Considering India's digital infrastructure strengths like Aadhaar and UPI alongside its AI investment limitations, what is its realistic potential in AI-powered tourism?",
        "reference": "India's potential lies in leapfrogging traditional stages by coupling AI with its unique public-digital infrastructure like Aadhaar and UPI, which no other country matches in scale. However, it is held back by modest AI investment of $1.25 billion compared to China's multibillion initiatives, uneven broadband, regulatory uncertainties, and a large unorganized accommodation sector. The coming years will determine how effectively India translates its policy vision into on-the-ground solutions."
    },
    {
        "user_input": "How does the role of government policy differ in driving AI tourism adoption between Singapore, UAE, and India?",
        "reference": "Singapore uses its National AI Strategy 2.0 and Smart Nation policies to actively partner with tech firms like OpenAI through the STB MOU, enabling rapid system-wide implementation due to its compact geography. UAE drives adoption through the AI 2031 strategy backed by sovereign investment, achieving innovations like citywide contactless hotel check-in. India relies on broader national programs like Digital India and IIDP but faces slower ground-level implementation due to infrastructure gaps and regulatory uncertainties, making its approach more aspirational than operational compared to Singapore and UAE."
    },
    {
        "user_input": "What does a 5% improvement in demand prediction mean financially for hotels?",
        "reference": "A 5% improvement in demand prediction can increase hotel revenue by 2 to 3%."
    },
    {
        "user_input": "What is the Digistan plan in India?",
        "reference": "The Digistan plan is India's digital heritage initiative aimed at creating 3D models of monuments, though as of 2025 it remains only partially implemented."
    },
    {
        "user_input": "How does China leverage scale in its AI tourism strategy compared to smaller economies like Singapore?",
        "reference": "China leverages data from hundreds of millions of trips to train AI models at a scale no smaller economy can match, leading in AI publications and patents and applying AI across landmarks with facial recognition and smart city integration. Singapore compensates for its smaller scale through strategic national coordination, high digital penetration, and compact geography that allows rapid system-wide implementation through partnerships like the STB-OpenAI MOU. Both succeed but through fundamentally different approaches — China through volume, Singapore through precision."
    },
    {
        "user_input": "What are the key research gaps identified in the paper?",
        "reference": "The paper identifies five gaps: lack of systematic comparative analysis of AI tourism adoption across countries, limited empirical data quantifying actual ROI and benefits, absence of research on integrated multi-AI platforms, insufficient ethical and policy guidance especially for culturally diverse regions, and sparse understanding of specific user segments like rural versus urban Indian tourists."
    },
]

testset_df = pd.DataFrame(test_data)
print(f"Testset: {len(testset_df)} samples")
testset_df

Testset: 15 samples (all grounded in PDF)


,user_input,reference
0,What was the projected size of the global AI i...,The global AI in travel market was approximate...
1,How much has India pledged for AI investment c...,"India pledged $1.25 billion for AI investment,..."
2,Which country leads global AI investment and b...,The United States leads global AI investment w...
3,What AI robot did Japan showcase at Expo 2025 ...,"Japan showcased Miraia Lynx, an AI concierge r..."
4,What does a 5% improvement in demand predictio...,A 5% improvement in demand prediction can incr...
5,What is the Digistan plan and what is its curr...,The Digistan plan is India's digital heritage ...
6,How many international tourist arrivals did In...,India recorded approximately 18.89 million int...
7,What percentage of Japanese travelers use gene...,Around 80% of Japanese travelers use generativ...
8,What contactless innovation did Dubai implemen...,Dubai implemented a citywide contactless hotel...
9,What partnership did Singapore Tourism Board s...,"In July 2025, the Singapore Tourism Board sign..."


### Run Pipeline on All Test Questions

In [33]:
test_questions = testset_df["user_input"].tolist()
ground_truths = testset_df["reference"].tolist()

responses = []
retrieved_contexts = []

for question in tqdm(test_questions, desc="Running Advanced RAG (final)"):
    result = advanced_rag_query(
        query=question,
        k=20,
        rerank_top_k=5,
        use_rrf=True,
        n_query_variations=3,
        verbose=False,
    )
    responses.append(result["answer"])
    retrieved_contexts.append(result["sources"])

print(f"\nCompleted {len(responses)} queries")

Running Advanced RAG (final): 100%|██████████| 15/15 [02:06<00:00,  8.45s/it]


Completed 15 queries


## Step 13: Ragas Evaluation

In [36]:
eval_list = [
    {"user_input": q, "response": r, "retrieved_contexts": c, "reference": g}
    for q, r, c, g in zip(test_questions, responses, retrieved_contexts, ground_truths)
]

evaluation_dataset = EvaluationDataset.from_list(eval_list)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

print("Running Ragas evaluation...")
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), AnswerRelevancy()],
    llm=evaluator_llm,
)

print("\n" + "-" * 50)
print("Ragas Evaluation Results:")
print(result)
print("=" * 50)

/tmp/ipykernel_5349/3898511382.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))


Running Ragas evaluation...


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]


Ragas Evaluation Results (final):
{'context_recall': 0.9867, 'faithfulness': 0.9891, 'factual_correctness(mode=f1)': 0.8200, 'answer_relevancy': 0.9034}


In [37]:
result_df = result.to_pandas()
print(result_df[["user_input", "context_recall", "faithfulness", "factual_correctness(mode=f1)", "answer_relevancy"]].to_string())

                                                                                                           user_input  context_recall  faithfulness  factual_correctness(mode=f1)  answer_relevancy
0              What was the projected size of the global AI in travel market in 2024 and 2029, and what was its CAGR?             1.0      1.000000                          0.86          0.966616
1                                                     How much has India pledged for AI investment compared to China?             1.0      1.000000                          0.86          0.953238
2           Which country leads global AI investment and by how much, according to the Stanford AI Index Report 2024?             1.0      1.000000                          0.67          0.987487
3                                            What AI robot did Japan showcase at Expo 2025 Osaka and what does it do?             1.0      1.000000                          0.89          0.911278
4                   